# Token Count Statistics

Computes token-count statistics across the three model result CSVs (`OutputTokenCombined_GPT4o.csv`, `OutputTokenCombined_Qwen.csv`, `OutputTokenCombined_GPTOSS.csv`), grouped by **model** and **context_variant** (minimal / method / class).

For each file we track `output_token`, `input_token`, and `total_token` (`input_token + output_token`, since both consume API cost):

1. Token count per file (per model, context_variant)
2. Token count per bump_id / instance (per model, context_variant) — sum of all files in that bump
3. Mean token count per file, per model-context_variant — `sum(total_token) / FILES_PER_VARIANT`
4. Mean and median token count per instance, per model-context_variant — mean is `sum(per-bump total_token) / BUMPS_PER_VARIANT`; median is the median of the per-bump totals

In [20]:
import pandas as pd
from pathlib import Path

In [21]:
BASE = Path.cwd()

CSV_FILES = [
    BASE / "OutputTokenCombined_GPT4o.csv",
    BASE / "OutputTokenCombined_Qwen.csv",
    BASE / "OutputTokenCombined_GPTOSS.csv",
]

# Expected size of every (model, context_variant) group. Every variant is run
# against the same 89 bumps / 5790 test files, so these should never vary
# across models or variants — the assertions below fail loudly if they ever do.
FILES_PER_VARIANT = 5790
BUMPS_PER_VARIANT = 89

## Load detail rows

Each CSV has per-file detail rows followed by aggregate footer sections (`PER-BUMP AGGREGATES`, `PER-TEST-FILE AGGREGATES`) separated by a blank line. We only need the detail rows here, so we stop reading at the first blank line. `total_token` is added as `output_token + input_token`.

In [22]:
def load_detail_rows(csv_path: Path) -> pd.DataFrame:
    """Load only the per-file detail rows, stopping before the aggregate footer."""
    detail_lines = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        header = f.readline().strip().split(",")
        for line in f:
            if line.strip() == "" or line.strip() == ",".join([""] * len(header)):
                break
            detail_lines.append(line.rstrip("\n").split(","))

    df = pd.DataFrame(detail_lines, columns=header)
    df["output_token"] = pd.to_numeric(df["output_token"], errors="coerce")
    df["input_token"] = pd.to_numeric(df["input_token"], errors="coerce")
    return df


dfs = [load_detail_rows(p) for p in CSV_FILES if p.exists()]
df_all = pd.concat(dfs, ignore_index=True)
df_all["total_token"] = df_all["output_token"] + df_all["input_token"]
print(df_all.shape)
df_all.head()

(52110, 7)


,model,context_variant,bump_id,file_name,output_token,input_token,total_token
0,GPT4o,minimal,BBC01,BBC01U2Test_prompt.txt,142,858,1000
1,GPT4o,minimal,BBC02,BBC02U0Test_prompt.txt,388,1195,1583
2,GPT4o,minimal,BBC02,BBC02U1Test_prompt.txt,397,834,1231
3,GPT4o,minimal,BBC03,BBC03U0Test_prompt.txt,254,1195,1449
4,GPT4o,minimal,BBC03,BBC03U1Test_prompt.txt,206,834,1040


### Sanity check group sizes

Confirms every (model, context_variant) group has exactly `FILES_PER_VARIANT` files and `BUMPS_PER_VARIANT` unique bumps before we rely on those constants as divisors below.

In [23]:
file_counts = df_all.groupby(["model", "context_variant"]).size()
bump_counts = df_all.groupby(["model", "context_variant"])["bump_id"].nunique()

assert (file_counts == FILES_PER_VARIANT).all(), f"Unexpected file counts:\n{file_counts}"
assert (bump_counts == BUMPS_PER_VARIANT).all(), f"Unexpected bump counts:\n{bump_counts}"
assert df_all[["output_token", "input_token"]].isna().sum().sum() == 0, "Found missing token counts"

print(f"OK: every group has {FILES_PER_VARIANT} files across {BUMPS_PER_VARIANT} bumps, no missing values.")

OK: every group has 5790 files across 89 bumps, no missing values.


## 1. Token count per file (per model, context_variant)

This is just the detail-level data: one row per generated test file, with its output/input/total token counts, labeled by model and context_variant.

In [24]:
df_per_file = df_all[
    ["model", "context_variant", "bump_id", "file_name", "output_token", "input_token", "total_token"]
].sort_values(["model", "context_variant", "bump_id", "file_name"]).reset_index(drop=True)
df_per_file.to_csv(BASE / "TokenStats_PerFile.csv", index=False)
df_per_file.head()

,model,context_variant,bump_id,file_name,output_token,input_token,total_token
0,GPT4o,class,BBC01,BBC01U2Test_prompt.txt,342,1918,2260
1,GPT4o,class,BBC02,BBC02U0Test_prompt.txt,340,9989,10329
2,GPT4o,class,BBC02,BBC02U1Test_prompt.txt,464,9628,10092
3,GPT4o,class,BBC03,BBC03U0Test_prompt.txt,336,9989,10325
4,GPT4o,class,BBC03,BBC03U1Test_prompt.txt,469,9628,10097


## 2. Token count per bump_id / instance (per model, context_variant)

Sums the output/input/total tokens of every file belonging to the same bump_id (i.e. total tokens for that instance).

In [25]:
df_per_bump = df_all.groupby(
    ["model", "context_variant", "bump_id"], as_index=False
)[["output_token", "input_token", "total_token"]].sum()
df_per_bump.to_csv(BASE / "TokenStats_PerBump.csv", index=False)
df_per_bump.head()

,model,context_variant,bump_id,output_token,input_token,total_token
0,GPT4o,class,BBC01,342,1918,2260
1,GPT4o,class,BBC02,804,19617,20421
2,GPT4o,class,BBC03,805,19617,20422
3,GPT4o,class,BBC04,52857,121328,174185
4,GPT4o,class,BBC05,1447,6371,7818


## 3. Mean token count per file, per model-context_variant

`mean = sum(total_cost_per_file) / FILES_PER_VARIANT` for each of output/input/total, computed explicitly rather than relying on an implicit groupby mean.

In [26]:
summary_per_file = df_all.groupby(["model", "context_variant"])[["output_token", "input_token", "total_token"]].sum() / FILES_PER_VARIANT
summary_per_file = summary_per_file.rename(columns={
    "output_token": "mean_output_token_per_file",
    "input_token": "mean_input_token_per_file",
    "total_token": "mean_total_token_per_file",
})
summary_per_file.to_csv(BASE / "TokenStats_MeanPerFile.csv")
summary_per_file

mean_output_token_per_file  \
model            context_variant                               
GPT4o            class                            413.139896   
                 method                           377.942660   
                 minimal                          224.512953   
GPT_OSS_120b     class                            639.642314   
                 method                           603.901554   
                 minimal                          383.761658   
Qwen3_480b_cloud class                            489.305872   
                 method                           571.593782   
                 minimal                          378.033333   

                                  mean_input_token_per_file  \
model            context_variant                              
GPT4o            class                          3907.995337   
                 method                         1247.578756   
                 minimal                         894.282729   
GPT_OSS_120b     class                          3907.995337   
                 method                         1247.578756   
                 minimal                         894.282729   
Qwen3_480b_cloud class                          3886.328497   
                 method                         1263.968048   
                 minimal                         909.321416   

                                  mean_total_token_per_file  
model            context_variant                             
GPT4o            class                          4321.135233  
                 method                         1625.521416  
                 minimal                        1118.795682  
GPT_OSS_120b     class                          4547.637651  
                 method                         1851.480311  
                 minimal                        1278.044387  
Qwen3_480b_cloud class                          4375.634370  
                 method                         1835.561831  
                 minimal                        1287.354750

## 4. Mean and median token count per instance (bump), per model-context_variant

Starting from the per-bump totals computed in step 2 (`df_per_bump`, one row per bump = sum of `total_cost_per_file` for that bump):

- `mean = sum(per-bump total) / BUMPS_PER_VARIANT`, computed explicitly
- `median` = median of the per-bump totals (no fixed-divisor equivalent, computed directly)

In [27]:
grouped_bump = df_per_bump.groupby(["model", "context_variant"])[["output_token", "input_token", "total_token"]]

mean_per_instance = grouped_bump.sum() / BUMPS_PER_VARIANT
median_per_instance = grouped_bump.median()

summary_per_instance = mean_per_instance.join(median_per_instance, lsuffix="_mean", rsuffix="_median")
summary_per_instance.to_csv(BASE / "TokenStats_MeanMedianPerInstance.csv")
summary_per_instance

output_token_mean  input_token_mean  \
model            context_variant                                        
GPT4o            class                 26877.303371     254239.247191   
                 method                24587.505618      81162.707865   
                 minimal               14605.955056      58178.617978   
GPT_OSS_120b     class                 41612.685393     254239.247191   
                 method                39287.528090      81162.707865   
                 minimal               24966.067416      58178.617978   
Qwen3_480b_cloud class                 31832.370787     252829.685393   
                 method                37185.707865      82228.932584   
                 minimal               24593.404494      59156.977528   

                                  total_token_mean  output_token_median  \
model            context_variant                                          
GPT4o            class               281116.550562               2972.0   
                 method              105750.213483               3802.0   
                 minimal              72784.573034               2086.0   
GPT_OSS_120b     class               295851.932584               5818.0   
                 method              120450.235955               5925.0   
                 minimal              83144.685393               3780.0   
Qwen3_480b_cloud class               284662.056180               5065.0   
                 method              119414.640449               5956.0   
                 minimal              83750.382022               3148.0   

                                  input_token_median  total_token_median  
model            context_variant                                          
GPT4o            class                       23370.0             27583.0  
                 method                      10777.0             15352.0  
                 minimal                     10012.0             11942.0  
GPT_OSS_120b     class                       23370.0             29732.0  
                 method                      10777.0             17068.0  
                 minimal                     10012.0             11883.0  
Qwen3_480b_cloud class                       23459.0             30641.0  
                 method                      10911.0             18113.0  
                 minimal                     10136.0             12937.0